In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================================
# CITATION INFORMATION
# ============================================================
# If you find this code useful for your research, please cite:
#
# Muharrem BALCI, STATISTICAL RELIABILITY AND EXPLAINABILITY
# OF MODERN CONVNEXTV2 AND SWIN TRANSFORMER ARCHITECTURES IN THE
# CLASSIFICATION OF MULTIPLE RETINAL DISEASES BASED ON FUNDUS IMAGES,
# (Submitted for publication), 2026.
#
# GitHub: https://github.com/mblci/Retina-Diseases-DeepLearning-Benchmark
# ============================================================

# ============================================================
# ADVANCED STATISTICAL ANALYSIS AND MODEL EXPLAINABILITY PIPELINE
# ============================================================
# Description: This script performs a comprehensive evaluation of
# trained models, including statistical significance tests,
# model efficiency metrics, and visual interpretability using Grad-CAM.
#
# Key Features:
# 1. Class Distribution Analysis (Visualization & CSV)
# 2. Performance Evaluation (Acc, F1, Precision, Recall)
# 3. Comparative Visualization (Confusion Matrix & ROC Panels)
# 4. Statistical Testing (McNemar Tests & Bootstrap Confidence Intervals)
# 5. Efficiency Metrics (FLOPs, Parameters, Inference Throughput)
# 6. Visual Explainability (Grad-CAM heatmaps for all architectures)
# 7. Detailed Error Pattern Analysis
# 8. Automated PDF Reporting
#
# Dataset Source: [Eye Disease Image Dataset: https://data.mendeley.com/datasets/s9bfhswzjb/1]
# ============================================================


import os, sys, warnings, time, json, random
from datetime import datetime
from collections import Counter
from itertools import combinations
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from PIL import Image
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, auc,
    precision_score, recall_score, f1_score, accuracy_score
)
from sklearn.preprocessing import label_binarize
from statsmodels.stats.contingency_tables import mcnemar

import timm
from ptflops import get_model_complexity_info
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from fpdf import FPDF

warnings.filterwarnings('ignore')

# --------------------------------------------------------------------------------
# 1. CONFIGURATION AND PATHS (GÜNCELLENEN KISIM BURASI)
# --------------------------------------------------------------------------------
# Modellerin 1. kod tarafından kaydedildiği klasör yolu ile eşitlendi.
MODEL_DIR = "./results/dynamic_models_metrics"

MODEL_PATHS = {
    "ConvNeXtV2_Base": os.path.join(MODEL_DIR, "best_ConvNeXtV2_Base.pth"),
    "EfficientNetV2_S": os.path.join(MODEL_DIR, "best_EfficientNetV2_S.pth"),
    "Swin_Tiny": os.path.join(MODEL_DIR, "best_Swin_Tiny.pth")
}

# Modellerin varlık kontrolü
missing_models = []
for name, path in MODEL_PATHS.items():
    if not os.path.exists(path):
        missing_models.append(name)

if missing_models:
    print(f"❌ HATA: Şu modeller bulunamadı: {missing_models}")
    print(f"Lütfen önce 1. Notebook'u çalıştırarak modelleri '{MODEL_DIR}' klasörüne kaydedin.")
    # Stop execution if running in a notebook (opsiyonel)
    # sys.exit()
else:
    print("✅ Tüm modeller başarıyla doğrulandı. Analiz başlıyor...")

TEST_DIR = "./dataset/split_data/test"
OUT_DIR = "./results/ultimate_analysis_report"
IMG_SIZE = 224
BATCH_SIZE = 32
device = "cuda" if torch.cuda.is_available() else "cpu"

# Klasör yapısını oluşturma
sub_dirs = ['tables', 'plots_combined', 'gradcam', 'error_analysis', 'distribution']
for sub in sub_dirs:
    os.makedirs(os.path.join(OUT_DIR, sub), exist_ok=True)

# Sınıf tespiti
temp_ds = datasets.ImageFolder(TEST_DIR)
CLASS_NAMES = temp_ds.classes
num_classes = len(CLASS_NAMES)
print(f" Classes Detected: {CLASS_NAMES}")

# --------------------------------------------------------------------------------
# 2. HELPER FUNCTIONS
# --------------------------------------------------------------------------------
def build_and_load(model_key, path):
    model_list = {
        "ConvNeXtV2_Base": "convnextv2_base",
        "EfficientNetV2_S": "tf_efficientnetv2_s.in21k_ft_in1k",
        "Swin_Tiny": "swin_tiny_patch4_window7_224"
    }
    model = timm.create_model(model_list[model_key], pretrained=False, num_classes=num_classes)

    # State dict yükleme işlemi
    sd = torch.load(path, map_location=device)
    if 'state_dict' in sd: sd = sd['state_dict']

    # Model katman isimlerini eşleme (Gerekiyorsa prefix temizleme)
    new_sd = {k.replace('module.', ''): v for k, v in sd.items()}
    model.load_state_dict(new_sd, strict=True)

    return model.to(device).eval()


# --------------------------------------------------------------------------------
# 3. DATA DISTRIBUTION ANALYSIS
# --------------------------------------------------------------------------------
print("\n Analyzing data distribution...")
counts = Counter(temp_ds.targets)
df_dist = pd.DataFrame({"Class": CLASS_NAMES, "Count": [counts[i] for i in range(num_classes)]})
df_dist.to_csv(os.path.join(OUT_DIR, 'distribution', 'test_distribution.csv'), index=False)

plt.figure(figsize=(10, 6))
sns.barplot(x="Class", y="Count", data=df_dist, palette="viridis")
plt.title("Test Set Class Distribution")
plt.savefig(os.path.join(OUT_DIR, 'distribution', 'test_distribution.png'))
plt.close()

# --------------------------------------------------------------------------------
# 4. PREDICTION COLLECTION
# --------------------------------------------------------------------------------
transform_eval = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
test_loader = DataLoader(datasets.ImageFolder(TEST_DIR, transform=transform_eval), batch_size=BATCH_SIZE, shuffle=False)

results_storage = {}
y_true_all = []
img_paths = [s[0] for s in temp_ds.samples]

for m_name, m_path in MODEL_PATHS.items():
    print(f"\n Testing {m_name}...")
    model = build_and_load(m_name, m_path)
    y_true, y_pred, y_prob = [], [], []

    with torch.no_grad():
        for imgs, lbls in tqdm(test_loader):
            imgs = imgs.to(device)
            out = model(imgs)
            prob = torch.softmax(out, dim=1)
            y_true.extend(lbls.numpy())
            y_pred.extend(torch.argmax(prob, dim=1).cpu().numpy())
            y_prob.extend(prob.cpu().numpy())

    results_storage[m_name] = {
        "y_pred": np.array(y_pred),
        "y_prob": np.array(y_prob),
        "correct": (np.array(y_pred) == np.array(y_true)).astype(int)
    }
    y_true_all = np.array(y_true)

# --------------------------------------------------------------------------------
# 5. COMPARATIVE PANELS (Confusion Matrix & ROC)
# --------------------------------------------------------------------------------
print("\n Generating Panel Plots...")

# Combined Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(24, 7))
for i, (m_name, data) in enumerate(results_storage.items()):
    cm = confusion_matrix(y_true_all, data['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    axes[i].set_title(f"{m_name} (Acc: {accuracy_score(y_true_all, data['y_pred'])*100:.2f}%)")
plt.savefig(os.path.join(OUT_DIR, 'plots_combined', 'combined_confusion_matrices.png'))
plt.close()

# Combined ROC Curves
fig, axes = plt.subplots(1, 3, figsize=(24, 7))
y_true_bin = label_binarize(y_true_all, classes=range(num_classes))
for i, (m_name, data) in enumerate(results_storage.items()):
    for j in range(num_classes):
        fpr, tpr, _ = roc_curve(y_true_bin[:, j], data['y_prob'][:, j])
        axes[i].plot(fpr, tpr, label=f'{CLASS_NAMES[j]} (AUC={auc(fpr, tpr):.3f})')
    axes[i].plot([0, 1], [0, 1], 'k--')
    axes[i].set_title(f"ROC: {m_name}"); axes[i].legend(loc='lower right')
plt.savefig(os.path.join(OUT_DIR, 'plots_combined', 'combined_roc_curves.png'))
plt.close()

# --------------------------------------------------------------------------------
# 6. STATISTICAL TESTS (McNemar & Bootstrap)
# --------------------------------------------------------------------------------
print("\n Performing statistical analyses...")

# McNemar Significance Test
mcn_results = []
names = list(MODEL_PATHS.keys())
for m1, m2 in combinations(names, 2):
    c1, c2 = results_storage[m1]['correct'], results_storage[m2]['correct']
    n10 = np.sum((c1==1) & (c2==0))
    n01 = np.sum((c1==0) & (c2==1))
    p_val = mcnemar([[0, n10], [n01, 0]], exact=True).pvalue
    mcn_results.append({"Comparison": f"{m1} vs {m2}", "p-value": p_val, "Significant": p_val < 0.05})
pd.DataFrame(mcn_results).to_csv(os.path.join(OUT_DIR, 'tables', 'mcnemar_results.csv'), index=False)

# Bootstrap Accuracy Difference
def get_bootstrap_ci(y_true, p1, p2, n_iter=1000):
    diffs = []
    for _ in range(n_iter):
        idx = np.random.choice(len(y_true), len(y_true), replace=True)
        acc1 = np.mean(p1[idx] == y_true[idx])
        acc2 = np.mean(p2[idx] == y_true[idx])
        diffs.append(acc1 - acc2)
    return np.percentile(diffs, [2.5, 97.5])

boot_results = []
for m1, m2 in combinations(names, 2):
    ci = get_bootstrap_ci(y_true_all, results_storage[m1]['y_pred'], results_storage[m2]['y_pred'])
    boot_results.append({"Pair": f"{m1}-{m2}", "CI_Lower": ci[0], "CI_Upper": ci[1]})
pd.DataFrame(boot_results).to_csv(os.path.join(OUT_DIR, 'tables', 'bootstrap_ci.csv'), index=False)

# --------------------------------------------------------------------------------
# 7. MODEL COMPLEXITY AND GRAD-CAM
# --------------------------------------------------------------------------------
print("\n Grad-CAM and Complexity Analysis...")
complexity_data = []
sample_indices = [np.where(y_true_all == i)[0][0] for i in range(num_classes)]

for m_name in MODEL_PATHS.keys():
    model = build_and_load(m_name, MODEL_PATHS[m_name])

    # Complexity Info
    flops, params = get_model_complexity_info(model, (3, 224, 224), as_strings=True, print_per_layer_stat=False)

    # Inference Throughput
    dummy = torch.randn(1, 3, 224, 224).to(device)
    start = time.time()
    for _ in range(100): _ = model(dummy)
    tpt = 100 / (time.time() - start)

    complexity_data.append({"Model": m_name, "Params": params, "FLOPs": flops, "Throughput(img/s)": round(tpt, 2)})

    # Grad-CAM Visualization
    target_lyr = get_target_layer(model, m_name)
    cam = GradCAM(model=model, target_layers=[target_lyr])
    m_grad_dir = os.path.join(OUT_DIR, 'gradcam', m_name)
    os.makedirs(m_grad_dir, exist_ok=True)

    for idx in sample_indices:
        input_img = transform_eval(Image.open(img_paths[idx]).convert('RGB')).unsqueeze(0).to(device)
        grayscale_cam = cam(input_tensor=input_img, targets=None)[0, :]
        rgb_img = np.array(Image.open(img_paths[idx]).convert('RGB').resize((224, 224))).astype(np.float32) / 255
        cam_viz = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)
        cv2.imwrite(os.path.join(m_grad_dir, f"{CLASS_NAMES[y_true_all[idx]]}_cam.png"), cv2.cvtColor(cam_viz, cv2.COLOR_RGB2BGR))

pd.DataFrame(complexity_data).to_csv(os.path.join(OUT_DIR, 'tables', 'model_complexity.csv'), index=False)

# --------------------------------------------------------------------------------
# 8. ERROR ANALYSIS
# --------------------------------------------------------------------------------
print("\n Performing error analysis...")
for m_name in names:
    data = results_storage[m_name]
    wrong_idx = np.where(data['correct'] == 0)[0]
    m_err_dir = os.path.join(OUT_DIR, 'error_analysis', m_name)
    os.makedirs(m_err_dir, exist_ok=True)

    if len(wrong_idx) > 0:
        err_df = pd.DataFrame({"True": [CLASS_NAMES[y_true_all[i]] for i in wrong_idx],
                               "Pred": [CLASS_NAMES[data['y_pred'][i]] for i in wrong_idx]})
        err_df.value_counts().to_csv(os.path.join(m_err_dir, 'error_patterns.csv'))

# --------------------------------------------------------------------------------
# 9. PDF REPORT GENERATION
# --------------------------------------------------------------------------------
try:
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", 'B', 16)
    pdf.cell(200, 10, "Retinal Disease Classification Analysis Report", ln=True, align='C')
    pdf.set_font("Arial", size=12)
    pdf.ln(10)
    pdf.cell(200, 10, f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}", ln=True)
    pdf.ln(5)

    pdf.cell(200, 10, "1. Performance Summary", ln=True, align='L')
    for m_name in names:
        acc = accuracy_score(y_true_all, results_storage[m_name]['y_pred'])
        pdf.cell(200, 8, f" - {m_name}: Accuracy = {acc:.4f}", ln=True)

    pdf.ln(10)
    pdf.cell(200, 10, "2. Statistical Significance (McNemar)", ln=True)
    for res in mcn_results:
        pdf.cell(200, 8, f" - {res['Comparison']}: p-value = {res['p-value']:.4e} (Sig: {res['Significant']})", ln=True)

    pdf.output(os.path.join(OUT_DIR, "Final_Analysis_Report.pdf"))
    print("\n📄 PDF Report Prepared.")
except Exception as e:
    print(f" PDF Report Error: {e}")

# Performance Table Export
perf_final = []
for m_name in names:
    y_p = results_storage[m_name]['y_pred']
    perf_final.append({
        "Model": m_name,
        "Accuracy": accuracy_score(y_true_all, y_p),
        "F1-Macro": f1_score(y_true_all, y_p, average='macro'),
        "Precision": precision_score(y_true_all, y_p, average='macro'),
        "Recall": recall_score(y_true_all, y_p, average='macro')
    })
pd.DataFrame(perf_final).to_csv(os.path.join(OUT_DIR, 'tables', 'overall_performance.csv'), index=False)

print(f"\n ALL ANALYSES COMPLETED SUCCESSFULLY!")
print(f" Results saved to: {OUT_DIR}")